<a href="https://colab.research.google.com/github/Patcharaporn-Anajakpob/683020588-2-Essential-Data-Science-69/blob/main/SimulteData_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##ver.1

In [ ]:
import random
import pandas as pd
import sqlite3

class Customer:
    def __init__(self, customer_id, name, age):
        self.customer_id = customer_id
        self.name = name
        self.age = age


class Policy:
    def __init__(self, policy_id, insurance_type, risk_level):
        self.policy_id = policy_id
        self.insurance_type = insurance_type
        self.risk_level = risk_level


class Claim:
    def __init__(self, claim_amount, status):
        self.claim_amount = claim_amount
        self.status = status

def calculate_premium(risk_level):
    if risk_level == "Low":
        return random.randint(5000,10000)
    elif risk_level == "Medium":
        return random.randint(10001,15000)
    else:
        return random.randint(15001,25000)


def generate_coverage():
    return random.randint(300000,1000000)


# default argument
def create_status(status_list=["Approved","Rejected","Pending"]):
    return random.choice(status_list)

insurance_data = []
insurance_types = ["Car","Health","Life","Home"]
risk_levels = ["Low","Medium","High"]

for i in range(300):

    customer = Customer(
        f"C{i+1:03d}",
        f"Customer_{i+1}",
        random.randint(18,70)
    )

    policy = Policy(
        f"P{i+1:03d}",
        random.choice(insurance_types),
        random.choice(risk_levels)
    )

    premium = calculate_premium(policy.risk_level)

    coverage = generate_coverage()

    claim = Claim(
        random.randint(0,coverage),
        create_status()
    )
    insurance_data.append([
        customer.customer_id,
        customer.name,
        customer.age,
        policy.policy_id,
        policy.insurance_type,
        policy.risk_level,
        premium,
        coverage,
        claim.claim_amount,
        claim.status
    ])

columns=[
"Customer_ID",
"Customer_Name",
"Age",
"Policy_ID",
"Insurance_Type",
"Risk_Level",
"Premium",
"Coverage",
"Claim_Amount",
"Claim_Status"
]

df=pd.DataFrame(insurance_data,columns=columns)
df.to_csv("insurance.csv",index=False)

conn=sqlite3.connect("insurance.db")

df.to_sql(
    "insurance",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

In [ ]:
df.head(300)

,Customer_ID,Customer_Name,Age,Policy_ID,Insurance_Type,Risk_Level,Premium,Coverage,Claim_Amount,Claim_Status
0,C001,Customer_1,18,P001,Car,Low,9046,473993,83301,Rejected
1,C002,Customer_2,32,P002,Car,Low,9209,858006,761737,Pending
2,C003,Customer_3,41,P003,Car,High,18649,904033,216644,Rejected
3,C004,Customer_4,29,P004,Life,Low,5351,906346,802416,Approved
4,C005,Customer_5,30,P005,Home,Low,6560,500927,134620,Pending
...,...,...,...,...,...,...,...,...,...,...
295,C296,Customer_296,66,P296,Car,Medium,14337,327814,127897,Approved
296,C297,Customer_297,55,P297,Car,Low,7797,833133,199451,Rejected
297,C298,Customer_298,68,P298,Life,Low,6705,924955,385771,Approved
298,C299,Customer_299,49,P299,Home,Low,9371,650477,466138,Rejected


##ver.2

In [7]:
import random
import pandas as pd
import sqlite3


# =========================================================
# 1. CLASS
# =========================================================

class Customer:
    def __init__(self, customer_id, name, age, occupation):
        self.customer_id = customer_id
        self.name = name
        self.age = age
        self.occupation = occupation


class Policy:
    def __init__(self, policy_id, insurance_type, plan, risk_level,
                 premium, coverage):
        self.policy_id = policy_id
        self.insurance_type = insurance_type
        self.plan = plan
        self.risk_level = risk_level
        self.premium = premium
        self.coverage = coverage


class Claim:
    def __init__(self, claim_amount, reason, status):
        self.claim_amount = claim_amount
        self.reason = reason
        self.status = status


# =========================================================
# 2. รายชื่อคนไทย
# =========================================================

first_names = [
    "สมชาย", "สมหญิง", "กิตติ", "วิชัย", "ธนกร",
    "ปกรณ์", "ธีรภัทร", "พีรพล", "อนุชา", "ภัทร",
    "ณัฐพล", "ศุภชัย", "อานนท์", "เอกชัย", "วรพล",
    "ปวีณา", "อรทัย", "ศิริพร", "ณัฐชา", "พิมพ์ชนก",
    "ชลธิชา", "สุภาวดี", "กนกวรรณ", "ปาริชาติ", "นภัสสร"
]

last_names = [
    "ใจดี", "สุขใจ", "บุญมี", "ทองดี", "ศรีสุข",
    "ชัยมงคล", "วงศ์ไทย", "แก้วมณี", "พรหมมา", "อินทร์ทอง",
    "บุญรอด", "รุ่งเรือง", "ศิริกุล", "อุดมทรัพย์", "วัฒนา",
    "แสงทอง", "เจริญสุข", "สุวรรณ", "ธรรมดี", "มงคลชัย"
]


# =========================================================
# 3. อาชีพ
# =========================================================

occupations = [
    "นักศึกษา",
    "พนักงานออฟฟิศ",
    "ครู",
    "แพทย์",
    "คนขับรถ",
    "เกษตรกร",
    "เจ้าของธุรกิจ",
    "พนักงานโรงงาน",
    "พนักงานก่อสร้าง",
    "ตำรวจ"
]


# =========================================================
# 4. ประเภทประกัน
# =========================================================

insurance_types = [
    "Car",
    "Health",
    "Life",
    "Home"
]


# =========================================================
# 5. กำหนดระดับความเสี่ยง
#    โดยดูจาก "อาชีพ + ประเภทประกัน"
# =========================================================

def determine_risk(occupation, insurance_type):

    # -------------------------
    # ประกันรถยนต์
    # -------------------------

    if insurance_type == "Car":

        if occupation in [
            "นักศึกษา",
            "พนักงานออฟฟิศ",
            "ครู",
            "แพทย์"
        ]:
            return random.choice(["Low", "Low", "Medium"])

        elif occupation in [
            "คนขับรถ",
            "เกษตรกร",
            "เจ้าของธุรกิจ"
        ]:
            return random.choice(["Medium", "Medium", "High"])

        else:
            return random.choice(["Medium", "High"])


    # -------------------------
    # ประกันสุขภาพ
    # -------------------------

    elif insurance_type == "Health":

        if occupation in [
            "นักศึกษา",
            "พนักงานออฟฟิศ",
            "ครู"
        ]:
            return random.choice(["Low", "Low", "Medium"])

        elif occupation in [
            "แพทย์",
            "เจ้าของธุรกิจ"
        ]:
            return random.choice(["Medium", "Medium", "High"])

        elif occupation in [
            "คนขับรถ",
            "เกษตรกร",
            "พนักงานโรงงาน",
            "พนักงานก่อสร้าง",
            "ตำรวจ"
        ]:
            return random.choice(["Medium", "High", "High"])

        else:
            return "Medium"


    # -------------------------
    # ประกันชีวิต
    # -------------------------

    elif insurance_type == "Life":

        if occupation in [
            "นักศึกษา",
            "พนักงานออฟฟิศ",
            "ครู"
        ]:
            return random.choice(["Low", "Low", "Medium"])

        elif occupation in [
            "แพทย์",
            "เจ้าของธุรกิจ"
        ]:
            return random.choice(["Medium", "Medium", "High"])

        elif occupation in [
            "คนขับรถ",
            "เกษตรกร",
            "พนักงานโรงงาน",
            "พนักงานก่อสร้าง",
            "ตำรวจ"
        ]:
            return random.choice(["Medium", "High", "High"])

        else:
            return "Medium"


    # -------------------------
    # ประกันบ้าน
    # -------------------------

    else:

        if occupation in [
            "นักศึกษา",
            "พนักงานออฟฟิศ",
            "ครู",
            "แพทย์"
        ]:
            return random.choice(["Low", "Low", "Medium"])

        elif occupation in [
            "เกษตรกร",
            "เจ้าของธุรกิจ"
        ]:
            return random.choice(["Medium", "Medium", "High"])

        else:
            return random.choice(["Low", "Medium"])


# =========================================================
# 6. กำหนดแผนประกัน
#
# รูปแบบเดียวกัน → วงเงินเท่ากัน
# แต่เบี้ยสามารถต่างกันตาม Risk Level
# =========================================================

def create_policy_plan(insurance_type, risk_level):

    plans = {

        "Car": {
            "Basic": (500000, 8000),
            "Standard": (800000, 14000),
            "Premium": (1200000, 20000)
        },

        "Health": {
            "Basic": (300000, 10000),
            "Standard": (700000, 18000),
            "Premium": (1500000, 28000)
        },

        "Life": {
            "Basic": (500000, 10000),
            "Standard": (1000000, 18000),
            "Premium": (2000000, 30000)
        },

        "Home": {
            "Basic": (1000000, 8000),
            "Standard": (2000000, 14000),
            "Premium": (3000000, 20000)
        }
    }

    # สุ่มแผน
    plan = random.choice(
        list(plans[insurance_type].keys())
    )

    coverage, base_premium = plans[insurance_type][plan]


    # ความเสี่ยงมีผลต่อเบี้ย
    if risk_level == "Low":
        multiplier = 1.00

    elif risk_level == "Medium":
        multiplier = 1.15

    else:
        multiplier = 1.35


    premium = int(base_premium * multiplier)

    # ปัดเป็นหลักพัน
    premium = round(premium / 1000) * 1000

    return plan, premium, coverage


# =========================================================
# 7. สาเหตุการเคลม
# =========================================================

def generate_claim_reason(insurance_type):

    reasons = {

        "Car": [
            "รถชน",
            "รถถูกเฉี่ยวชน",
            "กระจกแตก",
            "น้ำท่วมรถ",
            "ไฟไหม้รถ"
        ],

        "Health": [
            "เจ็บป่วย",
            "เข้ารับการรักษา",
            "ผ่าตัด",
            "อุบัติเหตุ"
        ],

        "Life": [
            "เสียชีวิต",
            "ทุพพลภาพ",
            "อุบัติเหตุร้ายแรง"
        ],

        "Home": [
            "น้ำท่วม",
            "ไฟไหม้",
            "โจรกรรม",
            "พายุ",
            "ภัยธรรมชาติ"
        ]
    }

    return random.choice(reasons[insurance_type])


# =========================================================
# 8. จำนวนเงินเคลม
#
# ไม่สุ่มมั่ว
# แต่ขึ้นอยู่กับประเภทประกัน + สาเหตุ
# และห้ามเกิน Coverage
# =========================================================

def generate_claim_amount(insurance_type, coverage, reason):

    if insurance_type == "Car":

        ranges = {
            "รถชน": (10000, 150000),
            "รถถูกเฉี่ยวชน": (5000, 50000),
            "กระจกแตก": (5000, 30000),
            "น้ำท่วมรถ": (20000, 150000),
            "ไฟไหม้รถ": (100000, 500000)
        }


    elif insurance_type == "Health":

        ranges = {
            "เจ็บป่วย": (5000, 50000),
            "เข้ารับการรักษา": (10000, 100000),
            "ผ่าตัด": (50000, 300000),
            "อุบัติเหตุ": (10000, 200000)
        }


    elif insurance_type == "Life":

        if reason == "เสียชีวิต":
            return coverage

        elif reason == "ทุพพลภาพ":
            return round((coverage * 0.5) / 1000) * 1000

        else:
            return round(
                random.uniform(
                    coverage * 0.25,
                    coverage * 0.75
                ) / 1000
            ) * 1000


    else:  # Home

        ranges = {
            "น้ำท่วม": (30000, 500000),
            "ไฟไหม้": (100000, 1000000),
            "โจรกรรม": (20000, 300000),
            "พายุ": (30000, 300000),
            "ภัยธรรมชาติ": (50000, 500000)
        }


    min_amount, max_amount = ranges[reason]

    # ห้ามเคลมเกิน Coverage
    max_amount = min(max_amount, coverage)

    # ป้องกันกรณี max < min
    min_amount = min(min_amount, max_amount)

    # จำนวนเงินเป็นหลักพัน
    amount = random.randrange(
        (min_amount // 1000) * 1000,
        (max_amount // 1000 + 1) * 1000,
        1000
    )

    return amount


# =========================================================
# 9. สร้างข้อมูลการเคลม
#
# Risk สูง → มีโอกาสเคลมมากกว่า
# =========================================================

def create_claim(risk_level, insurance_type, coverage):

    if risk_level == "Low":
        claim_probability = 0.10

    elif risk_level == "Medium":
        claim_probability = 0.20

    else:
        claim_probability = 0.35


    # ไม่มีการเคลม
    if random.random() > claim_probability:

        return Claim(
            0,
            "ไม่มีการเคลม",
            "No Claim"
        )


    # มีการเคลม
    reason = generate_claim_reason(
        insurance_type
    )

    amount = generate_claim_amount(
        insurance_type,
        coverage,
        reason
    )


    # สถานะ
    status = random.choices(
        ["Approved", "Pending", "Rejected"],
        weights=[75, 20, 5],
        k=1
    )[0]


    return Claim(
        amount,
        reason,
        status
    )


# =========================================================
# 10. สร้างข้อมูล 300 คน
# =========================================================

insurance_data = []


for i in range(300):

    # -------------------------
    # ลูกค้า
    # -------------------------

    name = (
        random.choice(first_names)
        + " "
        + random.choice(last_names)
    )

    age = random.randint(20, 65)

    occupation = random.choice(
        occupations
    )


    customer = Customer(
        f"C{i+1:03d}",
        name,
        age,
        occupation
    )


    # -------------------------
    # ประเภทประกัน
    # -------------------------

    insurance_type = random.choice(
        insurance_types
    )


    # -------------------------
    # ความเสี่ยง
    # อาชีพ + ประเภทประกัน
    # -------------------------

    risk_level = determine_risk(
        occupation,
        insurance_type
    )


    # -------------------------
    # แผน + เบี้ย + Coverage
    # -------------------------

    plan, premium, coverage = create_policy_plan(
        insurance_type,
        risk_level
    )


    policy = Policy(
        f"P{i+1:03d}",
        insurance_type,
        plan,
        risk_level,
        premium,
        coverage
    )


    # -------------------------
    # การเคลม
    # -------------------------

    claim = create_claim(
        risk_level,
        insurance_type,
        coverage
    )


    # -------------------------
    # เก็บข้อมูล
    # -------------------------

    insurance_data.append([

        customer.customer_id,
        customer.name,
        customer.age,
        customer.occupation,

        policy.policy_id,
        policy.insurance_type,
        policy.plan,
        policy.risk_level,

        policy.premium,
        policy.coverage,

        claim.claim_amount,
        claim.reason,
        claim.status
    ])


# =========================================================
# 11. สร้าง DataFrame
# =========================================================

columns = [

    "Customer_ID",
    "Customer_Name",
    "Age",
    "Occupation",

    "Policy_ID",
    "Insurance_Type",
    "Policy_Plan",
    "Risk_Level",

    "Premium",
    "Coverage",

    "Claim_Amount",
    "Claim_Reason",
    "Claim_Status"
]


df = pd.DataFrame(
    insurance_data,
    columns=columns
)


# =========================================================
# 12. ตรวจสอบจำนวนข้อมูล
# =========================================================

print("จำนวนแถวและคอลัมน์:", df.shape)

print("\nจำนวนลูกค้า:", len(df))

print("\nตัวอย่างข้อมูล:")
display(df.head(300))


# =========================================================
# 13. บันทึกเป็น CSV
# =========================================================

df.to_csv(
    "insurance.csv",
    index=False
)


# =========================================================
# 14. บันทึกเป็น SQLite
# =========================================================

conn = sqlite3.connect(
    "insurance.db"
)

df.to_sql(
    "insurance",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("\nสร้างไฟล์ insurance.csv และ insurance.db เรียบร้อยแล้ว")

จำนวนแถวและคอลัมน์: (300, 13)

จำนวนลูกค้า: 300

ตัวอย่างข้อมูล:


,Customer_ID,Customer_Name,Age,Occupation,Policy_ID,Insurance_Type,Policy_Plan,Risk_Level,Premium,Coverage,Claim_Amount,Claim_Reason,Claim_Status
0,C001,กนกวรรณ วัฒนา,26,พนักงานออฟฟิศ,P001,Life,Premium,Low,30000,2000000,0,ไม่มีการเคลม,No Claim
1,C002,ชลธิชา ทองดี,39,แพทย์,P002,Health,Basic,Medium,12000,300000,0,ไม่มีการเคลม,No Claim
2,C003,อนุชา ศิริกุล,60,แพทย์,P003,Car,Basic,Medium,9000,500000,166000,ไฟไหม้รถ,Approved
3,C004,ณัฐชา ชัยมงคล,49,แพทย์,P004,Car,Basic,Medium,9000,500000,0,ไม่มีการเคลม,No Claim
4,C005,ศิริพร พรหมมา,31,เจ้าของธุรกิจ,P005,Home,Basic,Medium,9000,1000000,0,ไม่มีการเคลม,No Claim
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,C296,อนุชา พรหมมา,51,คนขับรถ,P296,Life,Premium,High,40000,2000000,0,ไม่มีการเคลม,No Claim
296,C297,ศุภชัย ธรรมดี,33,พนักงานก่อสร้าง,P297,Health,Premium,High,38000,1500000,275000,ผ่าตัด,Approved
297,C298,พิมพ์ชนก สุวรรณ,22,คนขับรถ,P298,Car,Standard,Medium,16000,800000,12000,รถชน,Approved
298,C299,นภัสสร วงศ์ไทย,31,ตำรวจ,P299,Car,Premium,Medium,23000,1200000,0,ไม่มีการเคลม,No Claim



สร้างไฟล์ insurance.csv และ insurance.db เรียบร้อยแล้ว
